### Multimodal LLM for QC

#### Context
- [Reference paper](https://arxiv.org/pdf/2509.24888)
- [Codebase](https://github.com/EvolvingLMMs-Lab/LLaVA-OneVision-1.5)
- [pretrained model](https://huggingface.co/lmms-lab/LLaVA-OneVision-1.5-4B-stage0)
- [Interactive HuggingFace Dash](https://huggingface.co/spaces/lmms-lab/LLaVA-OneVision-1.5)
- [Example dataset](https://openneuro.org/datasets/ds004173/versions/1.0.2)

#### Notes
- Needs large RAM and possibly GPU

#### imports

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM
from qwen_vl_utils import process_vision_info
import torch

In [ ]:
model_path = "lmms-lab/LLaVA-OneVision-1.5-4B-Instruct"

# Load the model on the available device(s)
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype="auto", device_map="auto", trust_remote_code=True
)

# default processor
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda" if torch.cuda.is_available() else "cpu")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=1024)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)